In [2]:
from pathlib import Path
from itertools import product
import hashlib
import inspect
import json
import os

import numpy as np
import pandas as pd
from IPython.display import display
from deeprootgen.data_model import RootSimulationModel
from deeprootgen.model import RootSystemSimulation
from deeprootgen.statistics.spatial_rld import calculate_core_rld

In [3]:
# Update INFILE for your checkout. The reference notebooks use ../../data/grapevine/.
INFILE = Path('..', '..', 'data', 'grapevine', 'wong_2026_rld.csv')
if not INFILE.exists() and Path('wong_2026_rld.csv').exists():
    INFILE = Path('wong_2026_rld.csv')
if not INFILE.exists():
    raise FileNotFoundError(f'Cannot find Wong 2026 observations: {INFILE}')

BASE_SEED = 42
CORE_HEIGHT_CM = 10.0  # Assumed 10-cm thick interval, inferred from z_mid_cm spacing.
N_TEST_CORES = 2
N_REPLICATES = 5
LENGTH_SOURCE = 'geometric'  # Matches the geometric segment convention in Fichtl notebooks.

# Wong's independent unit for site-specific spatial validation is the core profile.
# Do not interpret the site management grouping as a grapevine rootstock or timepoint.
SITE = 'Site 1'

# The original Fichtl objective is spatial (0.60), root-order (0.25), length (0.15).
# Wong has diameter classes, not branching orders: substitute a diameter-profile loss.
OBJECTIVE_MODE = 'rld'  # 'composite' enables the much slower diameter-bin calculations.
LOSS_WEIGHTS = {'spatial': 0.60, 'diameter': 0.25, 'length': 0.15}
if OBJECTIVE_MODE not in {'rld', 'composite'}:
    raise ValueError('OBJECTIVE_MODE must be rld or composite.')
if not np.isclose(sum(LOSS_WEIGHTS.values()), 1.0):
    raise ValueError('LOSS_WEIGHTS must sum to 1.')
if CORE_HEIGHT_CM <= 0 or N_REPLICATES < 1:
    raise ValueError('CORE_HEIGHT_CM and N_REPLICATES must be positive.')

# Baseline starting values are adapted from the Fichtl notebooks. They are NOT fitted Wong estimates.
# Wider initial lengths acknowledge that Wong cores extend further laterally/deeply.
BASE_PARAMETERS = dict(
    random_seed=BASE_SEED,
    max_order=5,
    root_ratio=0.5,
    fine_root_threshold=0.15,
    outer_root_num=5,
    inner_root_num=5,
    min_primary_length=15.0,
    max_primary_length=100.0,
    base_diameter=0.5,
    diameter_reduction=0.2,
    apex_diameter=0.05,
    min_sec_root_num=1,
    max_sec_root_num=4,
    growth_sec_root=0.1,
    min_sec_root_length=3.0,
    max_sec_root_length=40.0,
    segments_per_root=8,
    length_reduction=0.7,
    root_vary=10.0,
    interbranch_distance=2.0,
    mechanical_constraints=0.0,
    root_tissue_density=0.15,
    gravitropism=0.5,
    t=1,
    enable_soil=False,
    soil_layer_height=10,
    soil_n_layers=16,
    soil_layer_width=10,
    soil_n_cols=32,
)

In [4]:
DIAMETER_COLS = sorted(
    [c for c in pd.read_csv(INFILE, nrows=0).columns
     if c.startswith('grapevine_length_cm_') and c.endswith('_mm')],
    key=lambda c: float(c.removeprefix('grapevine_length_cm_').split('_')[0]),
)

def diameter_limits(column):
    labels = column.removeprefix('grapevine_length_cm_').removesuffix('_mm').split('_')
    if len(labels) != 2:
        raise ValueError(f'Invalid diameter class: {column}')
    return float(labels[0]), float(labels[1])

DIAMETER_BINS = [(c, *diameter_limits(c)) for c in DIAMETER_COLS]
if not DIAMETER_BINS or any(lo >= hi for _, lo, hi in DIAMETER_BINS):
    raise ValueError('Missing or invalid diameter classes.')

def load_observations(path=INFILE):
    data = pd.read_csv(path)
    required = {
        'site', 'core_id', 'management_grouping', 'x_cm', 'y_cm', 'z_mid_cm',
        'sampled', 'core_volume_cm3', 'grapevine_rld', *DIAMETER_COLS,
    }
    missing = sorted(required.difference(data.columns))
    if missing:
        raise ValueError(f'Missing required fields: {missing}')
    if data.duplicated(['site', 'core_id', 'z_mid_cm']).any():
        raise ValueError('Duplicate site-core-depth records.')
    if not data['sampled'].isin(['Y', 'N']).all():
        raise ValueError('Unrecognised sampled values.')
    if data['core_volume_cm3'].le(0).any():
        raise ValueError('Nonpositive soil core volumes.')
    if data['grapevine_rld'].lt(0).any():
        raise ValueError('Negative root length density.')
    # Unsampled records have zero placeholders; do not fit those records.
    data = data.loc[data['sampled'].eq('Y')].copy()
    if data.empty or data[list(required - {'site', 'core_id', 'management_grouping', 'sampled'})].isna().any().any():
        raise ValueError('Missing values among sampled observations.')
    # The core volume is for an entire 10 cm sampled depth interval, not a 10-cm voxel.
    data['core_radius_cm'] = np.sqrt(data['core_volume_cm3'] / (np.pi * CORE_HEIGHT_CM))
    # DeepRootGen's z axis is negative below ground. z_mid_cm is a positive depth.
    data['z_lower_bound'] = -(data['z_mid_cm'] + CORE_HEIGHT_CM / 2)
    data['z_upper_bound'] = -(data['z_mid_cm'] - CORE_HEIGHT_CM / 2)
    data['observed_length_cm'] = data['grapevine_rld'] * data['core_volume_cm3']
    return data.reset_index(drop=True)

observed_df = load_observations()
print('Sampled records:', len(observed_df), 'Sites:', observed_df.site.nunique(),
      'Core profiles:', observed_df.groupby(['site', 'core_id']).ngroups)

def split_cores(site_observations, seed=BASE_SEED, n_test=N_TEST_CORES):
    cores = np.array(sorted(site_observations['core_id'].unique()))
    if not 0 < n_test < len(cores):
        raise ValueError('N_TEST_CORES must be between 1 and n_cores - 1.')
    permuted = np.random.default_rng(seed).permutation(cores)
    return sorted(permuted[n_test:].tolist()), sorted(permuted[:n_test].tolist())


def site_observations(site):
    result = observed_df.loc[observed_df.site.eq(site)].copy()
    if result.empty:
        raise ValueError(f'Unknown site: {site}')
    # A core profile must have fixed horizontal coordinates.
    if (result.groupby('core_id')[['x_cm', 'y_cm']].nunique() > 1).any().any():
        raise ValueError('Horizontal coordinates vary with depth within a core.')
    return result.reset_index(drop=True)


def sha256_file(path):
    sha = hashlib.sha256()
    with open(path, 'rb') as fh:
        for piece in iter(lambda: fh.read(2**20), b''):
            sha.update(piece)
    return sha.hexdigest()


def configuration_signature(payload):
    # Include source and dataset identity to prevent mixing checkpoint results
    # after changing source, objective, seeds or input observations.
    payload = {
        **payload,
        'data_sha256': sha256_file(INFILE),
        'simulator_sha256': sha256_file(inspect.getfile(RootSystemSimulation)),
        'core_rld_sha256': sha256_file(inspect.getfile(calculate_core_rld)),
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:12]

Sampled records: 880 Sites: 5 Core profiles: 70


In [5]:
from joblib import Parallel, delayed, parallel_config

OUT_DIR = Path('outputs/wong_2026_grid_search')
OUT_DIR.mkdir(parents=True, exist_ok=True)
GROUPS_TO_FIT = ['Site 1']  # Set None to fit all five sites independently.
N_JOBS = min(6, os.cpu_count() or 1)
RESUME_SEARCH = True
CHECKPOINT_EVERY = 5

# Exploratory one-shot design, not final calibrated ranges.
PARAMETER_GRID = {
    'max_primary_length': [100.0, 160.0],
    'outer_root_num': [4, 6],
    'max_sec_root_num': [3, 5],
    'max_sec_root_length': [30.0, 60.0],
}

In [6]:
def predict_cores(parameters, site_data, seed, *, need_diameters=False,
                  return_architecture=False):
    """One simulated RSA per site and seed, evaluated at every selected sampled core.

    Crucially, this does NOT generate an independent plant for every core/depth.
    """
    model = RootSimulationModel(**{**parameters, 'random_seed': int(seed)})
    simulation = RootSystemSimulation(random_seed=int(seed))
    simulation.run(model)
    nodes, edges = simulation.G.as_df()

    # For composite losses, cumulative diameter thresholds are evaluated as
    # real root lengths in the same cylinder. Bin lengths are differences.
    thresholds = sorted({d for _, low, high in DIAMETER_BINS for d in (low, high) if d > 0})
    records = []
    for row in site_data.itertuples(index=False):
        kwargs = dict(
            x=float(row.x_cm), y=float(row.y_cm),
            z_lower_bound=float(row.z_lower_bound),
            z_upper_bound=float(row.z_upper_bound),
            core_radius=float(row.core_radius_cm),
            length_source=LENGTH_SOURCE,
        )
        rld = calculate_core_rld(nodes, **kwargs)
        result = dict(site=row.site, core_id=row.core_id, z_mid_cm=row.z_mid_cm,
                      sim_rld=float(rld),
                      sim_length_cm=float(rld * row.core_volume_cm3))
        if need_diameters:
            cumulative = {0.0: 0.0}
            for threshold in thresholds:
                # Diameter classes are treated as [lower, upper): include values
                # exactly on a bin's lower boundary rather than its upper.
                strict_upper = float(np.nextafter(threshold, -np.inf))
                cumulative[threshold] = calculate_core_rld(
                    nodes, diameter_max_mm=strict_upper, **kwargs
                ) * row.core_volume_cm3
            for column, lower, upper in DIAMETER_BINS:
                result['sim_' + column] = max(
                    0.0, float(cumulative[upper] - cumulative[lower])
                )
        records.append(result)

    prediction = pd.DataFrame(records)
    if return_architecture:
        return prediction, {'seed': int(seed), 'nodes': nodes, 'edges': edges}
    return prediction


def score_prediction(observations, predictions, *, objective_mode=OBJECTIVE_MODE):
    """Compare only genuinely sampled core records, preserving spatial zeros.

    Spatial loss is normalised against a zero-prediction benchmark, as in Fichtl.
    Diameter profiles are optional and not required to sum to reported total RLD.
    """
    keys = ['site', 'core_id', 'z_mid_cm']
    comparison = observations.merge(predictions, on=keys, how='left', validate='one_to_one')
    if comparison['sim_rld'].isna().any():
        raise ValueError('Predictions do not cover every sampled site/core/depth.')
    actual = comparison['grapevine_rld'].to_numpy(float)
    simulated = comparison['sim_rld'].to_numpy(float)
    square = (simulated - actual)**2
    occupied = actual > 0
    if occupied.any() and (~occupied).any():
        mse = 0.5 * (square[occupied].mean() + square[~occupied].mean())
        zero_mse = 0.5 * np.mean(actual[occupied]**2)
    else:
        mse = square.mean()
        zero_mse = np.mean(actual**2)
    # All-zero observations have a zero benchmark loss. Use a squared-error
    # scale of at least 1 to avoid catastrophic division by 1e-12.
    spatial_loss = float(mse / max(zero_mse, 1.0 if not occupied.any() else 1e-12))
    observed_length = float(comparison['observed_length_cm'].sum())
    predicted_length = float(comparison['sim_length_cm'].sum())
    length_loss = float(((predicted_length - observed_length) /
                         max(observed_length, 1e-9))**2)
    if objective_mode == 'composite':
        a = comparison[DIAMETER_COLS].to_numpy(float)
        b = comparison[['sim_' + c for c in DIAMETER_COLS]].to_numpy(float)
        diameter_loss = float(((a - b)**2).sum() / max((a**2).sum(), 1e-12))
        objective = (LOSS_WEIGHTS['spatial'] * spatial_loss +
                     LOSS_WEIGHTS['diameter'] * diameter_loss +
                     LOSS_WEIGHTS['length'] * length_loss)
    elif objective_mode == 'rld':
        diameter_loss = float('nan')
        objective = spatial_loss
    else:
        raise ValueError(f'Unknown objective mode: {objective_mode}')
    comparison['rld_residual'] = comparison['sim_rld'] - comparison['grapevine_rld']
    return comparison, {
        'objective': float(objective),
        'rmse': float(np.sqrt(square.mean())),
        'spatial_loss': spatial_loss,
        'diameter_loss': diameter_loss,
        'length_loss': length_loss,
        'observed_length_cm': observed_length,
        'simulated_length_cm': predicted_length,
        'n_sampled': len(comparison),
        'observed_occupied': int(occupied.sum()),
        'simulated_occupied': int((simulated > 0).sum()),
    }


def evaluate_candidate(parameters, site_data, core_ids, *, test=False,
                       save_comparisons=False):
    """Evaluate one parameter candidate with common random numbers per site."""
    selected = site_data.loc[site_data.core_id.isin(core_ids)].copy()
    if selected.empty:
        raise ValueError('No sampled core observations for selected core IDs.')
    records, comparisons = [], []
    for replicate in range(N_REPLICATES):
        seed = BASE_SEED + replicate + (10_000_000 if test else 0)
        prediction = predict_cores(
            parameters, selected, seed,
            need_diameters=OBJECTIVE_MODE == 'composite',
        )
        comparison, metrics = score_prediction(selected, prediction)
        records.append(dict(replicate=replicate, seed=seed, **metrics))
        if save_comparisons:
            comparison['replicate'] = replicate
            comparison['seed'] = seed
            comparisons.append(comparison)
    return pd.DataFrame(records), comparisons

In [7]:
def save_checkpoint(rows, path):
    frame = (pd.DataFrame(rows).drop_duplicates('candidate_id', keep='last')
             .sort_values('candidate_id'))
    tmp = path.with_suffix('.csv.tmp')
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)


def grid_candidate(candidate_id, values, site, site_df, train_cores):
    candidate = dict(zip(PARAMETER_GRID.keys(), values))
    parameters = {**BASE_PARAMETERS, **candidate}
    scores, _ = evaluate_candidate(parameters, site_df, train_cores)
    return dict(candidate_id=int(candidate_id), site=site, **candidate,
                train_objective=float(scores.objective.mean()),
                train_rmse=float(scores.rmse.mean()),
                train_spatial_loss=float(scores.spatial_loss.mean()),
                train_length_loss=float(scores.length_loss.mean()),
                train_diameter_loss=float(scores.diameter_loss.mean())
                if OBJECTIVE_MODE == 'composite' else float('nan'),
                train_objective_sd=float(scores.objective.std(ddof=1)),
                n_training_simulations=len(scores))


def calibrate_site(site):
    site_df = site_observations(site)
    train_cores, test_cores = split_cores(site_df)
    candidate_values = list(product(*PARAMETER_GRID.values()))
    payload = dict(site=site, train_cores=train_cores, test_cores=test_cores,
                   parameter_grid=PARAMETER_GRID, base_parameters=BASE_PARAMETERS,
                   n_replicates=N_REPLICATES, base_seed=BASE_SEED,
                   core_height_cm=CORE_HEIGHT_CM,
                   objective_mode=OBJECTIVE_MODE, loss_weights=LOSS_WEIGHTS,
                   length_source=LENGTH_SOURCE)
    signature = configuration_signature(payload)
    prefix = site.replace(' ','_').lower()
    checkpoint = OUT_DIR / f'{prefix}_grid_checkpoint_{signature}.csv'
    completed = pd.read_csv(checkpoint).to_dict('records') if (RESUME_SEARCH and checkpoint.exists()) else []
    done_ids = {int(row['candidate_id']) for row in completed}
    pending = [(i,v) for i,v in enumerate(candidate_values) if i not in done_ids]
    print(f'{site}: {len(candidate_values)} candidates; {len(pending)} pending; '
          f'{N_REPLICATES} simulator replicates/candidate; train={train_cores}; test={test_cores}')
    if pending:
        with parallel_config(backend='loky', n_jobs=N_JOBS, inner_max_num_threads=1):
            stream = Parallel(return_as='generator_unordered', batch_size=1, verbose=5)(
                delayed(grid_candidate)(i, v, site, site_df, train_cores) for i,v in pending
            )
            for j, row in enumerate(stream, start=1):
                completed.append(row)
                if j % CHECKPOINT_EVERY == 0:
                    save_checkpoint(completed, checkpoint)
        save_checkpoint(completed, checkpoint)
    results = (pd.DataFrame(completed).drop_duplicates('candidate_id',keep='last')
               .sort_values(['train_objective','candidate_id']).reset_index(drop=True))
    if len(results) != len(candidate_values):
        raise RuntimeError('Grid search incomplete.')
    best = results.iloc[0]
    best_choice = {k: best[k].item() if isinstance(best[k],np.generic) else best[k]
                   for k in PARAMETER_GRID}
    best_parameters = {**BASE_PARAMETERS, **best_choice}
    train_scores,_ = evaluate_candidate(best_parameters,site_df,train_cores)
    test_scores, comparisons = evaluate_candidate(best_parameters, site_df, test_cores,
                                                   test=True, save_comparisons=True)
    test_comp = pd.concat(comparisons,ignore_index=True)
    results.to_csv(OUT_DIR/f'{prefix}_grid_search.csv',index=False)
    train_scores.to_csv(OUT_DIR/f'{prefix}_train_scores.csv',index=False)
    test_scores.to_csv(OUT_DIR/f'{prefix}_heldout_scores.csv',index=False)
    test_comp.to_csv(OUT_DIR/f'{prefix}_heldout_core_comparison.csv',index=False)
    pd.DataFrame([best_parameters]).to_csv(OUT_DIR/f'{prefix}_best_parameters.csv',index=False)
    (OUT_DIR/f'{prefix}_run_config.json').write_text(json.dumps(
        {**payload,'signature':signature,'best_parameters':best_parameters},indent=2,default=str))
    print(f'Best {site}: {best_choice}; mean held-out RMSE: {test_scores.rmse.mean():.5g}')
    display(results.head(10))
    return dict(search=results,best_parameters=best_parameters,
                train_scores=train_scores,test_scores=test_scores,comparison=test_comp)

sites_to_fit = sorted(observed_df.site.unique()) if GROUPS_TO_FIT is None else GROUPS_TO_FIT
calibration_results = {site: calibrate_site(site) for site in sites_to_fit}

Site 1: 16 candidates; 16 pending; 5 simulator replicates/candidate; train=['tube1', 'tube10', 'tube11', 'tube12', 'tube13', 'tube14', 'tube4', 'tube5', 'tube6', 'tube7', 'tube8', 'tube9']; test=['tube2', 'tube3']


[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   9 out of  16 | elapsed:   53.1s remaining:   41.3s
[Parallel(n_jobs=6)]: Done  13 out of  16 | elapsed:  1.0min remaining:   14.0s
[Parallel(n_jobs=6)]: Done  16 out of  16 | elapsed:  1.2min finished


Best Site 1: {'max_primary_length': 160.0, 'outer_root_num': 6, 'max_sec_root_num': 5, 'max_sec_root_length': 60.0}; mean held-out RMSE: 0.38323


,candidate_id,site,max_primary_length,outer_root_num,max_sec_root_num,max_sec_root_length,train_objective,train_rmse,train_spatial_loss,train_length_loss,train_diameter_loss,train_objective_sd,n_training_simulations
0,15,Site 1,160.0,6,5,60.0,0.993411,0.280145,0.993411,0.976298,NaN,0.003832,5
1,3,Site 1,100.0,4,5,60.0,0.994064,0.280238,0.994064,0.980510,NaN,0.002983,5
2,9,Site 1,160.0,4,3,60.0,0.994354,0.280279,0.994354,0.983134,NaN,0.001730,5
3,7,Site 1,100.0,6,5,60.0,0.994736,0.280333,0.994736,0.980626,NaN,0.001112,5
4,1,Site 1,100.0,4,3,60.0,0.995141,0.280390,0.995141,0.987235,NaN,0.002193,5
5,11,Site 1,160.0,4,5,60.0,0.995371,0.280422,0.995371,0.988806,NaN,0.002587,5
6,6,Site 1,100.0,6,5,30.0,0.995587,0.280452,0.995587,0.986621,NaN,0.004332,5
7,14,Site 1,160.0,6,5,30.0,0.996323,0.280556,0.996323,0.985713,NaN,0.002586,5
8,13,Site 1,160.0,6,3,60.0,0.996585,0.280593,0.996585,0.990380,NaN,0.003925,5
9,2,Site 1,100.0,4,5,30.0,0.996841,0.280629,0.996841,0.989402,NaN,0.001913,5
